# 04 Run Eval

Pipeline and Query Analyzer evaluation using `templates/evaluation_queries.csv`. Replace placeholder positives with real ground-truth ranges before reporting metrics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -r /content/multimodal-search-demo/requirements-colab.txt

In [ ]:
import os, sys
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path('/content/multimodal-search-demo')
sys.path.insert(0, str(PROJECT_ROOT))
os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    print('GEMINI_API_KEY is not set. Analyzer will use rule-based fallback.')

EVAL_QUERIES = PROJECT_ROOT / 'templates' / 'evaluation_queries.csv'
ADAPTER_PATH = '/content/drive/MyDrive/korean_cooking_shorts_dataset/siglip2_lora_qv_r16_best'

In [ ]:
import pandas as pd
from src.eval.run_eval import evaluate_queries, load_eval_queries
from src.eval.analyzer_eval import evaluate_analyzer
from src.index.qdrant_client import get_qdrant_client
from src.models.bge_encoder import BGEEncoder
from src.models.siglip_encoder import SigLIPEncoder

queries = load_eval_queries(EVAL_QUERIES)
detail, analyzer_summary = evaluate_analyzer(queries)
display(pd.DataFrame([analyzer_summary]))

client = get_qdrant_client()
bge = BGEEncoder()
siglip = SigLIPEncoder(adapter_path=ADAPTER_PATH)

rows = [evaluate_queries(queries, client, bge, siglip, mode) for mode in ['text-only', 'image-only', 'hybrid', 'unified']]
pd.DataFrame(rows)